# Průzkumná analýza dat 
## Načtení a základní přehled dat

Před zahájením samotné analýzy si načtu kompletní dataset telekomunikačních dat, abych ověřila strukturu tabulky, datové typy a klíčové identifikátory.

In [0]:
select * from workspace.default.telco


## Základní metriky a kontrola duplicit
Ověřím celkový počet záznamů v datasetu a ujistím se, že každé ID je unikátní.

In [0]:
SELECT 
  COUNT(*) AS celkovy_pocet_zakazniku,
  COUNT(DISTINCT customerID) AS pocet_unikatnich_zakazniku
FROM 
  workspace.default.telco;

_Výsledek ukazuje, že jsou data v pořádku, nikde se neduplikují._


## Analýza odchodu zákazníků
Podívám se, kolik zákazníků z celkového počtu odešlo a jaké je procentuální zastoupení.

In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS procento_z_celku
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

Databricks visualization. Run in Databricks to view.

_Tímto jsem zjistila, že odchodovost zákazníků má celkem dost vysoké číslo a přidala jsem jednoduchý koláčový graf._

## Vliv typu smlouvy na odchod zákazníků
Analyzuji, jaký vliv má smluvní závazek na míru odchodovosti. Předpokládám, že nejrizikovější budou smlouvy, které jsou na měsíční bázi.

In [0]:
SELECT 
  Contract,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY Contract), 2) AS procento_v_ramci_smlouvy
FROM 
  workspace.default.telco
GROUP BY 
  Contract, 
  Churn
ORDER BY 
  Contract, 
  Churn;

Databricks visualization. Run in Databricks to view.

_V kódu nepočítám procento z úplného celku, ale samostatně pro každý typ smlouvy, abych viděla kolik procent lidí odešlo z určité skupiny. Čili, když se podíváme na sloupec s procenty, tak vidíme že:_  

_&#45; u měsíčních smluv odejde 42,71 % zákazníků_  
_&#45; u ročních smluv je to 11,27 % zákazníků_  
_&#45; u dvouletých smluv to je 2,83 % zákazníků._

_Tady jsem přidala i sloupcový graf._

## Finanční analýza a loajálnost
Podívám se jestli zákazníci kteří odcházejí, platí v průměru vyšší měsíční poplatky a po jaké době od firmy odcházejí.


In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(AVG(MonthlyCharges), 2) AS prumerna_mesicni_platba,
  ROUND(AVG(tenure), 1) AS prumerna_doba_u_firmy_v_mesicich
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

_Zde lze říct, že cena hraje určitě roli v rozhodování k odchodu._
_Klienti, kteří odcházejí platí v průměru 74,44 a ti věrní jen 61,27. Čili to znamená, že odcházejí ti co mají vlastně služby dražší._
_Když se podíváme na dobu jakou u nás jsou, tak jde vidět že vydrží kolem roka a půl. Možná bychom měli tento termín pohlídat a zkusit je kontaktovat s nabídkou výhodnějších služeb?_